# Router Baseline

The goal of this notebook is to train and inspect a baseline guarded query router for the GQR benchmark.

The workflow focuses on:
- defining the four benchmark labels
- loading and normalizing in-domain and out-of-domain text data
- training an embedding-based Law/Finance/Health classifier
- rejecting uncertain queries as out-of-domain
- evaluating the final router with ID accuracy, OOD accuracy, and the GQR score
- saving the trained artifacts for later CLI or benchmark usage

The reusable implementation lives in `src/router`. This notebook keeps the experiment readable while still using the same code paths as the command-line interface.

## 1. Environment

The first cell prepares the notebook environment. It finds the project root, adds `src` to the Python path, configures logging, and sets the project-local cache directory used for downloaded datasets and embedding models.

In [ ]:
from __future__ import annotations

import logging
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from router.cache import configure_project_cache

DATA_CACHE_DIR = configure_project_cache()

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    datefmt="%H:%M:%S",
)

{"project_root": PROJECT_ROOT, "data_cache_dir": DATA_CACHE_DIR}

The output confirms the project root and the active cache location. Downloads are stored under `data/cache`, so repeated runs can reuse local files instead of downloading the same datasets and embedding model again.

## 2. Labels

The benchmark uses four integer labels. The first three labels are in-domain routing targets, while label `3` is reserved for out-of-domain queries that should be rejected by the router.

In [ ]:
from router.labels import (
    ALL_LABELS,
    DOMAIN_TO_LABEL,
    ID_LABELS,
    LABEL_TO_DOMAIN,
    OOD,
    normalize_label,
)

label_rows = [
    {"label": label, "domain": LABEL_TO_DOMAIN[label]}
    for label in ALL_LABELS
]
pd.DataFrame(label_rows)

This table is the complete label space used by the router. During training, labels `0`, `1`, and `2` are treated as in-domain classes; label `3` is used only when the model decides that a query does not belong to any supported domain.

In [ ]:
examples = ["law", "fintech", "healthcare", "other", "2"]
pd.DataFrame(
    {"input": value, "normalized_label": normalize_label(value)}
    for value in examples
)

The normalization helper accepts both integer labels and common text aliases. This is useful when local files use domain names such as `fintech`, `medical`, or `other` instead of the benchmark integers.

## 3. Configuration

All experiment settings are collected in one place. This makes it easy to switch between official GQR data and a local CSV/JSON/Parquet file, reduce sample sizes for a quick run, or try a different embedding model.

In [ ]:
MODEL_DIR = PROJECT_ROOT / "artifacts" / "router"

# Set these to local CSV/JSON/JSONL/Parquet paths to skip official GQR loading.
TRAIN_PATH = None
VALID_PATH = None
OOD_VALID_PATH = None

EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
THRESHOLD = 0.55
TARGET_ID_RECALL = 0.95
BATCH_SIZE = 64

# Use small numbers for quick smoke tests, or None for full splits.
MAX_TRAIN_SAMPLES = None
MAX_VALID_SAMPLES = None
MAX_OOD_VALID_SAMPLES = 1000
SKIP_OOD_VALIDATION = False

For quick smoke tests, set `MAX_TRAIN_SAMPLES` and `MAX_VALID_SAMPLES` to small values. For a full baseline run, leave them as `None` so the loader keeps all available examples.

## 4. Load Data

The training split contains the three in-domain classes: law, finance, and health. If the official GQR package cannot load its configured data, the project falls back to public Hugging Face datasets for the same three domains.

At this stage we only prepare train and validation splits. OOD examples are added later during evaluation if the validation split does not already contain them.

In [ ]:
from router.data import (
    DatasetSplit,
    load_builtin_ood_validation_dataset,
    load_gqr_ood_test_dataset,
    load_gqr_train_dataset,
    load_tabular_dataset,
)


def split_summary(name: str, split: DatasetSplit) -> dict[str, object]:
    counts = pd.Series(split.labels).map(LABEL_TO_DOMAIN).value_counts().to_dict()
    return {
        "split": name,
        "rows": len(split.texts),
        "law": counts.get("law", 0),
        "finance": counts.get("finance", 0),
        "health": counts.get("health", 0),
        "ood": counts.get("ood", 0),
    }


if TRAIN_PATH is not None:
    train_split = load_tabular_dataset(TRAIN_PATH)
    valid_split = load_tabular_dataset(VALID_PATH) if VALID_PATH else train_split
else:
    train_split, valid_split = load_gqr_train_dataset()

train_split = train_split.limit(MAX_TRAIN_SAMPLES)
valid_split = valid_split.limit(MAX_VALID_SAMPLES)

pd.DataFrame(
    [
        split_summary("train", train_split),
        split_summary("validation", valid_split),
    ]
).set_index("split")

The summary table checks whether the loaded data has the expected domain coverage. A normal in-domain training run should contain law, finance, and health examples, while OOD rows may still be zero at this point.

## 5. Train

The baseline router has two parts. First, a SentenceTransformer model converts each query into an embedding. Then a logistic-regression classifier predicts one of the three in-domain labels.

OOD detection is handled by a confidence threshold. If the best in-domain probability is below the threshold, the router returns label `3` instead of forcing a domain prediction.

In [ ]:
from router.model import DomainRouter

router = DomainRouter(
    embedding_model=EMBEDDING_MODEL,
    threshold=THRESHOLD,
)

router.fit(
    train_split.texts,
    train_split.labels,
    valid_texts=valid_split.texts,
    valid_labels=valid_split.labels,
    target_id_recall=TARGET_ID_RECALL,
    batch_size=BATCH_SIZE,
)

router.threshold

The value displayed above is the calibrated rejection threshold. Higher thresholds usually reject more queries as OOD, while lower thresholds route more queries into one of the three supported domains.

## 6. Evaluate

Evaluation follows the GQR setup: in-domain examples measure routing accuracy, while OOD examples measure rejection accuracy. The final GQR score is the harmonic mean of these two values, so the router needs to do both tasks well.

In [ ]:
from router.metrics import confusion_counts, evaluate_router


def load_ood_validation_split() -> DatasetSplit:
    if SKIP_OOD_VALIDATION:
        return DatasetSplit(texts=[], labels=[])

    if OOD_VALID_PATH is not None:
        return load_tabular_dataset(OOD_VALID_PATH).limit(MAX_OOD_VALID_SAMPLES)

    try:
        return load_gqr_ood_test_dataset().limit(MAX_OOD_VALID_SAMPLES)
    except Exception as exc:
        print(f"Falling back to built-in OOD examples: {type(exc).__name__}: {exc}")
        return load_builtin_ood_validation_dataset(
            max_samples=MAX_OOD_VALID_SAMPLES,
        )


if any(label == OOD for label in valid_split.labels):
    report_split = valid_split
else:
    report_split = valid_split.extend(load_ood_validation_split())

predictions = router.predict(report_split.texts, batch_size=BATCH_SIZE)
scores = evaluate_router(report_split.labels, predictions)

pd.DataFrame(
    [
        {
            "id_accuracy": scores.id_accuracy,
            "ood_accuracy": scores.ood_accuracy,
            "gqr_score": scores.gqr_score,
            "rows": len(report_split.texts),
            "ood_rows": sum(label == OOD for label in report_split.labels),
        }
    ]
)

The score table separates in-domain routing quality from OOD rejection quality. The GQR score is low if either side is weak, so it is more informative here than plain accuracy over the combined dataset.

In [ ]:
confusion_rows = [
    {
        "true_label": true,
        "true_domain": LABEL_TO_DOMAIN[true],
        "predicted_label": pred,
        "predicted_domain": LABEL_TO_DOMAIN[pred],
        "count": count,
    }
    for (true, pred), count in sorted(
        confusion_counts(report_split.labels, predictions).items()
    )
]

pd.DataFrame(confusion_rows)

The confusion table shows where the router makes mistakes. In-domain confusions indicate routing errors between supported domains, while predictions of `ood` for an in-domain true label indicate over-rejection.

## 7. Inspect Predictions

A few hand-written queries make the model behavior easier to interpret than aggregate metrics alone. The examples below cover the three in-domain areas and one clearly unrelated query.

In [ ]:
sample_queries = [
    "Can my landlord keep my deposit after I move out?",
    "How should I diversify my retirement portfolio?",
    "What are common symptoms of iron deficiency?",
    "How do I bake sourdough bread at home?",
]

prediction_details = router.predict_with_scores(sample_queries, batch_size=BATCH_SIZE)
pd.DataFrame(
    {
        "query": query,
        "label": prediction.label,
        "domain": LABEL_TO_DOMAIN[prediction.label],
        "confidence": round(prediction.confidence, 4),
    }
    for query, prediction in zip(sample_queries, prediction_details)
)

The prediction table shows both the final label and the confidence used for thresholding. A query can have a likely in-domain class internally but still be returned as `ood` when confidence is below the calibrated threshold.

## 8. Save And Reload

After training, the router artifacts are saved to `artifacts/router`. This includes the fitted classifier metadata and, once the embedding model has been loaded, a local copy of the embedding model for more reliable offline prediction.

In [ ]:
router.save(MODEL_DIR)
reloaded_router = DomainRouter.load(MODEL_DIR)

{
    "model_dir": str(MODEL_DIR),
    "smoke_prediction": reloaded_router.predict_one(
        "How do I appeal a denied insurance claim?"
    ),
}

The smoke prediction verifies that the saved artifacts can be loaded back into a fresh router object. This is the same path used later by CLI prediction and the scoring helper functions.

## 9. Scoring Function

The scoring helpers provide a benchmark-compatible interface. They hide model loading behind a small wrapper so external evaluation code can call a simple function that maps text to a label.

In [ ]:
from router.scoring import scoring_function, scoring_function_batch

single_label = scoring_function(
    "Can I sue for breach of contract?",
    model_dir=MODEL_DIR,
)
batch_labels = scoring_function_batch(
    [
        "Can I sue for breach of contract?",
        "What is a reasonable ETF allocation?",
    ],
    model_dir=MODEL_DIR,
    batch_size=BATCH_SIZE,
)

{"single_label": single_label, "batch_labels": batch_labels}

These helpers are the cleanest interface for benchmark-style evaluation. The single-text function is convenient for simple tests, while the batch variant is more efficient for larger scoring runs.

## 10. Optional Official GQR Score

The official benchmark can be run directly from the notebook when the `gqr` package is installed and the trained model is available. The flag is disabled by default so normal notebook runs do not unexpectedly trigger the full benchmark.

In [ ]:
RUN_OFFICIAL_GQR_SCORE = False

if RUN_OFFICIAL_GQR_SCORE:
    import gqr

    official_scores = gqr.score_batch(
        lambda batch: reloaded_router.predict(batch, batch_size=BATCH_SIZE),
        batch_size=BATCH_SIZE,
    )
    print(official_scores)
else:
    print("Set RUN_OFFICIAL_GQR_SCORE = True to run the official benchmark.")

When this flag is enabled, the official benchmark calls the trained router in batches. Keep it disabled while editing or smoke-testing the notebook, then enable it only for the final benchmark run.

## 11. Next Experiments

The baseline is intentionally simple, which makes it useful as a reference point. The most natural follow-up experiments are:

- Try a stronger embedding model.
- Tune `TARGET_ID_RECALL` and inspect the ID/OOD tradeoff.
- Add a larger local OOD validation file through `OOD_VALID_PATH`.
- Compare logistic regression with a different classifier over the same embeddings.